In [ ]:
import io

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

import brain_probability_flux.plotting as bpp

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
# This figure is dense enough that the project default (7pt) causes
# text/content overlap and excess empty space; use 5pt here instead.
plt.rcParams["font.size"] = 5.0

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT
panel_gap = bpp.PANEL_GAP_PT / 72  # gap between panels in the composite (in)

In [ ]:
# Bootstrapped entropy production rates and fractions of observed
# transitions for 2-10 clusters (scripts/bootstrap_epr.py), each
# (clusters, tasks, replicates)
data = bpp.load_results("epr_bootstrap", data_dir=DATA_DIR)
num_clusters_list = data["num_clusters"]
epr_data = data["eprs"]
fot_data = data["frac_observed_transitions"]

In [ ]:
task_names = bpp.TASK_NAMES_FULL
fig, axs = plt.subplots(
    1,
    5,
    figsize=(max_fig_width - 4 * panel_gap, max_fig_width / 4.0),
    width_ratios=[1, 1, 1, 1, 1],
)
task_colors = bpp.task_colors_stepped()
ax = axs[0]
for i in range(8):
    ax.errorbar(
        num_clusters_list,
        fot_data.mean(axis=2)[:, i],
        yerr=fot_data.std(axis=2)[:, i] * 2,
        ms=0,
        elinewidth=0.5,
        capsize=1,
        capthick=0.5,
        mew=0.5,
        marker="o",
        label=task_names[i],
        c=task_colors[i],
        alpha=0.8,
    )
ax.legend()
ax.axhline(1.0, c="k", ls="--")
ax.axhline(0.95, c="k", ls="--")
ax.set_xlabel("Number of cluster")
ax.set_ylabel("Fraction of observed transitions")

ax = axs[1]
for i in range(8):
    ax.errorbar(
        num_clusters_list,
        epr_data.mean(axis=2)[:, i],
        yerr=epr_data.std(axis=2)[:, i] * 2,
        ms=0,
        elinewidth=0.5,
        capsize=1,
        capthick=0.5,
        mew=0.5,
        marker="o",
        label=task_names[i],
        c=task_colors[i],
        alpha=0.8,
    )
ax.legend()
ax.set_xlabel("Number of cluster")
ax.set_ylabel("Entropy production rate (nats/s)")

ax = axs[2]
epr = epr_data[list(num_clusters_list).index(7)]
x = np.arange(epr.shape[0])
y = epr.mean(axis=1)
for i in range(len(x)):
    ax.bar(x[i], y[i], fill=False, edgecolor=task_colors[i], lw=0.5)
std = epr.std(axis=1)
for i in range(len(x)):
    ax.errorbar(
        x[i],
        y[i],
        yerr=2 * std[i],
        ms=0,
        elinewidth=0.5,
        capsize=1,
        capthick=0.5,
        mew=0.5,
        marker="o",
        fmt=".",
        c=task_colors[i],
    )
ax.set_xticks(np.arange(len(x)))
ax.set_xticklabels(task_names, rotation=90)
ax.set_ylabel("Entropy production rate (nats/s)")

ax = axs[3]
pvalue_map = bpp.pvalue_colormap()

num_tasks = len(task_names)
pvalue_matrix = np.zeros((num_tasks, num_tasks))
for i in range(num_tasks):
    for j in range(i + 1, num_tasks):
        results = stats.ks_2samp(epr[i], epr[j])
        p = results[1]
        pvalue_matrix[i, j] = p

log_pvalue_matrix = np.log10(pvalue_matrix)
mask = np.isfinite(log_pvalue_matrix)
vmin = np.min([np.min(log_pvalue_matrix[mask]), np.log10(0.01)])
divnorm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=np.log10(0.05), vmax=0)

cmap = pvalue_map
pd_log_pvalue_matrix = pd.DataFrame(
    log_pvalue_matrix, index=task_names, columns=task_names
)
ax = sns.heatmap(
    pd_log_pvalue_matrix,
    mask=~mask,
    cmap=cmap,
    linewidths=0.5,
    cbar_kws={"shrink": 0.5, "label": "$\\log_{{10}} p_\\mathrm{{KS}}$"},
    norm=divnorm,
    ax=ax,
)
ax.set_xticks(np.arange(num_tasks) + 0.5)
ax.set_xticklabels(task_names, rotation=90)

ax = axs[4]
response_rate = np.array([0, 0.26, 0.053, 0.025, 0.058, 0.159, 0.172, 0.477])
epr = epr_data[list(num_clusters_list).index(7)]
x = response_rate
y = epr.mean(axis=1)
std = epr.std(axis=1)
for i in range(len(x)):
    ax.errorbar(
        x[i],
        y[i],
        yerr=2 * std[i],
        lw=0,
        ms=1,
        elinewidth=0.5,
        capsize=1,
        capthick=0.5,
        mew=0.5,
        marker="o",
        c=task_colors[i],
    )
ax.set_xlabel("Response rate (s$^{-1}$)")
ax.set_ylabel("Entropy production rate (nats/s)")
results = stats.pearsonr(x, y)
r = results.statistic
p = results.pvalue
ax.text(
    0.04,
    0.95,
    f"$r={r:.3f}$\n$p={p:.3f}$",
    ha="left",
    va="top",
    fontsize=5,
    transform=ax.transAxes,
)

# ── Pass 1: measure axes height to compute square width for panel d ───────────
fig.savefig(io.BytesIO(), format="png")

fig_w_in = max_fig_width - 4 * panel_gap
fig_h_in = max_fig_width / 4.0
h_ax_in = axs[3].get_position().height * fig_h_in
W_ax_frac = sum(ax.get_position().width for ax in axs)

w_d_frac = h_ax_in / fig_w_in
w_other_frac = (W_ax_frac - w_d_frac) / 4
r_d = w_d_frac / w_other_frac

axs[0].get_gridspec().set_width_ratios([1, 1, 1, r_d, 1])

# Panel d includes its colorbar
colorbars = [ax for ax in fig.axes if ax not in axs]
panels = dict(zip("abcde", axs))
panels["d"] = [axs[3], *colorbars]
bpp.save_axes_panels(fig, panels, "../figures/fig-s17")